# Netflix Prize — Recommendation System

Pipeline: **bias baseline → biased matrix factorization (PyTorch/GPU) → evaluation (RMSE + MAP@10) → Top-K → ensemble blend**.

Runs on Colab (GPU) or locally. Set `DATA_DIR` in the config cell. Extension stubs (item-kNN, timeSVD++) at the bottom.

**Colab tip:** Runtime → Change runtime type → GPU. Use a High-RAM runtime for the full 100M-row parse.

## 0. Setup & config

In [ ]:
import os, glob, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- CONFIG ----------------------------------------------------------
# Folder with combined_data_*.txt + movie_titles.csv
#   Colab + Drive:  '/content/netflix'   |  Local: r'C:\\Users\\...\\netflix-prize\\data'
DATA_DIR   = '/content/netflix'      # <-- EDIT ME
CACHE_NPZ  = os.path.join(DATA_DIR, 'ratings_cache.npz')

SUBSAMPLE_FILES = [1, 2, 3, 4]   # which combined_data_N.txt to load
CHUNK_ROWS      = 5_000_000      # parse chunk size — bounds peak RAM

# Keep this fraction of USERS (each kept user keeps ALL their ratings — good for CF).
#   1.0  = full 100M ratings — use a HIGH-RAM Colab runtime.
#   0.35 = fits a standard ~12GB runtime; a defensible subset (problem statement allows it).
USER_FRAC = 1.0

N_FACTORS = 100
N_EPOCHS  = 20
LR        = 0.01               # Adam LR (reg is in the loss, not weight_decay — see MF cell)
REG       = 0.02               # L2 strength (lambda) added to the loss, Funk-SVD style
BATCH     = 100_000
SEED      = 42
TEST_FRAC = 0.20               # per-user most-recent holdout
RELEVANT_THRESHOLD = 3.5       # MAP@10 relevance (problem statement)

rng = np.random.default_rng(SEED)
print('config loaded')

In [ ]:
# --- Colab only: mount Drive and/or unzip the dataset. Skip if running locally. ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # Example: unzip the archive you uploaded to Drive into /content/netflix
    # !mkdir -p /content/netflix
    # !unzip -o '/content/drive/MyDrive/archive.zip' -d /content/netflix
except Exception as e:
    print('Not on Colab (or skip):', e)

## 1. Parse ratings → compact arrays (cached)

In [ ]:
def parse_combined(path, chunksize=CHUNK_ROWS):
    """Memory-bounded parse: stream in chunks, downcast immediately, drop strings.
    Format: a 'MovieID:' header line, then 'user,rating,date' rows.
    Returns int32 user, int16 movie (1-based), int8 rating, datetime64[D] date.
    Peak RAM is ~one chunk of strings instead of the whole file."""
    us, ms, rs, ds = [], [], [], []
    cur = 0
    for chunk in pd.read_csv(path, header=None, names=['user', 'rating', 'date'],
                             dtype={'user': str}, chunksize=chunksize):
        is_hdr = chunk['rating'].isna()                      # header lines -> NaN rating
        mv = pd.to_numeric(chunk['user'].where(is_hdr).str.rstrip(':'),
                           errors='coerce').ffill().fillna(cur)
        rows = ~is_hdr
        us.append(pd.to_numeric(chunk.loc[rows, 'user']).to_numpy(np.int32))
        ms.append(mv[rows].to_numpy(np.int16))
        rs.append(chunk.loc[rows, 'rating'].to_numpy(np.int8))
        ds.append(chunk.loc[rows, 'date'].to_numpy('datetime64[D]'))
        cur = int(mv.iloc[-1])
    return (np.concatenate(us), np.concatenate(ms),
            np.concatenate(rs), np.concatenate(ds))

if os.path.exists(CACHE_NPZ):
    z = np.load(CACHE_NPZ)
    raw_user, movie, rating, date = z['user'], z['movie'], z['rating'], z['date']
    print(f'loaded cache: {len(rating):,} ratings')
else:
    parts_u, parts_m, parts_r, parts_d = [], [], [], []
    for n in SUBSAMPLE_FILES:
        p = os.path.join(DATA_DIR, f'combined_data_{n}.txt')
        t = time.time()
        u, m, r, d = parse_combined(p)
        parts_u.append(u); parts_m.append(m); parts_r.append(r); parts_d.append(d)
        print(f'  parsed {os.path.basename(p)}: {len(r):,} rows in {time.time()-t:.0f}s')
    raw_user = np.concatenate(parts_u); movie = np.concatenate(parts_m)
    rating   = np.concatenate(parts_r); date  = np.concatenate(parts_d)
    del parts_u, parts_m, parts_r, parts_d
    np.savez(CACHE_NPZ, user=raw_user, movie=movie, rating=rating, date=date)
    print(f'parsed & cached {len(rating):,} ratings -> {CACHE_NPZ}')

In [ ]:
# Map sparse user IDs -> contiguous indices; optional user-subsample for tight RAM.
uniq_users, user = np.unique(raw_user, return_inverse=True)
user = user.astype(np.int32); del raw_user
movie = (movie - 1).astype(np.int16)            # 0-based

if USER_FRAC < 1.0:
    keep = np.zeros(len(uniq_users), bool)
    keep[rng.choice(len(uniq_users), int(len(uniq_users) * USER_FRAC), replace=False)] = True
    m_rows = keep[user]
    user, movie = user[m_rows], movie[m_rows]
    rating, date = rating[m_rows], date[m_rows]
    _, user = np.unique(user, return_inverse=True)   # re-densify kept user ids
    user = user.astype(np.int32)
    print(f'user-subsampled: {len(rating):,} ratings, {int(keep.sum()):,} users')

n_users  = int(user.max()) + 1
n_movies = int(movie.max()) + 1
print(f'{len(rating):,} ratings | {n_users:,} users | {n_movies:,} movies')

# Movie titles (latin-1; titles may contain commas)
titles = pd.read_csv(os.path.join(DATA_DIR, 'movie_titles.csv'),
                     header=None, encoding='latin-1', on_bad_lines='skip',
                     names=['movie_id', 'year', 'title'], usecols=[0, 1, 2])
title_of = dict(zip(titles['movie_id'].astype(int), titles['title']))
print('titles loaded:', len(title_of))

## 2. Exploratory Data Analysis

In [ ]:
sparsity = len(rating) / (n_users * n_movies)
print(f'Global mean rating : {rating.mean():.4f}')
print(f'Matrix density     : {sparsity:.4%}  (sparsity {1-sparsity:.4%})')

user_counts  = np.bincount(user,  minlength=n_users)
movie_counts = np.bincount(movie, minlength=n_movies)
print(f'Ratings/user  median {np.median(user_counts):.0f}  mean {user_counts.mean():.1f}  max {user_counts.max()}')
print(f'Ratings/movie median {np.median(movie_counts):.0f} mean {movie_counts.mean():.1f} max {movie_counts.max()}')

fig, ax = plt.subplots(1, 3, figsize=(15, 3.2))
vals, cnts = np.unique(rating, return_counts=True)
ax[0].bar(vals, cnts); ax[0].set_title('Rating distribution'); ax[0].set_xlabel('stars')
ax[1].hist(np.log10(user_counts + 1), bins=50); ax[1].set_title('log10 ratings/user')
ax[2].hist(np.log10(movie_counts + 1), bins=50); ax[2].set_title('log10 ratings/movie')
plt.tight_layout(); plt.show()

In [ ]:
# Temporal trend — the famous early-2004 mean-rating shift (motivates timeSVD++)
months = date.astype('datetime64[M]')
mdf = pd.DataFrame({'m': months, 'r': rating}).groupby('m')['r'].agg(['mean', 'count'])
fig, ax = plt.subplots(1, 2, figsize=(13, 3.2))
ax[0].plot(mdf.index, mdf['mean']); ax[0].set_title('Mean rating over time')
ax[1].plot(mdf.index, mdf['count']); ax[1].set_title('Ratings volume over time')
plt.tight_layout(); plt.show()
mdf.head()

## 3. Train / test split — per-user temporal holdout

Hold out each user's most recent `TEST_FRAC` ratings. Realistic (predict the future from the past) and leaves multiple held-out items per user for MAP@10. Users with `<5` ratings stay fully in train (cold-start, evaluated separately if desired).

In [ ]:
# Per-user temporal holdout — int32 intermediates, freed eagerly to keep RAM low.
order = np.lexsort((date, user))                       # by user, then date
su = user[order]
_, starts, counts = np.unique(su, return_index=True, return_counts=True)
starts = starts.astype(np.int32); counts = counts.astype(np.int32)
pos    = np.arange(len(order), dtype=np.int32) - np.repeat(starts, counts)
n_test = np.where(counts >= 5, np.ceil(counts * TEST_FRAC).astype(np.int32), 0)
thresh = np.repeat(counts - n_test, counts)
is_test = np.zeros(len(order), bool)
is_test[order] = pos >= thresh
del order, su, pos, thresh, starts, counts

tr = ~is_test
u_tr, m_tr, r_tr = user[tr], movie[tr], rating[tr]          # m_* stay int16
u_te, m_te, r_te = user[is_test], movie[is_test], rating[is_test]
print(f'train {tr.sum():,}  |  test {is_test.sum():,}')

## 4. Baseline bias model  `r̂ = μ + b_u + b_i`

In [ ]:
def fit_biases(u, m, r, n_u, n_m, reg=10.0, n_iter=10):
    mu = r.mean()
    bu = np.zeros(n_u, np.float32); bi = np.zeros(n_m, np.float32)
    uc = np.bincount(u, minlength=n_u); ic = np.bincount(m, minlength=n_m)
    for _ in range(n_iter):
        # item bias given user bias
        resid = r - mu - bu[u]
        bi = np.bincount(m, weights=resid, minlength=n_m) / (ic + reg)
        # user bias given item bias
        resid = r - mu - bi[m]
        bu = np.bincount(u, weights=resid, minlength=n_u) / (uc + reg)
    return mu, bu, bi

mu, bu, bi = fit_biases(u_tr, m_tr, r_tr, n_users, n_movies)

def predict_baseline(u, m):
    return np.clip(mu + bu[u] + bi[m], 1, 5)

def rmse(pred, actual):
    return float(np.sqrt(np.mean((pred - actual) ** 2)))

print('baseline test RMSE:', rmse(predict_baseline(u_te, m_te), r_te))

## 5. Biased Matrix Factorization (PyTorch, GPU)

`r̂ = μ + b_u + b_i + pᵤ · qᵢ` — the low-rank model, trained by SGD over observed entries.

**Regularization is in the loss, not Adam's `weight_decay`.** The classic Netflix λ≈0.02 is a *per-rating* SGD penalty (`p ← p + lr·(err·q − λ·p)`). Passing it as `weight_decay` over a **mean** MSE loss applies `λ·p` every step while the data gradient is only `~error/batch_size` — so decay overwhelms learning, every parameter freezes at 0, and RMSE sticks at the rating std (~1.08). Adding explicit L2 on the embeddings *used in each batch* keeps the penalty on the same (mean) scale as the data term, so it regularizes without freezing.

In [ ]:
import torch, torch.nn as nn
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

class MF(nn.Module):
    def __init__(self, n_u, n_m, k, mu):
        super().__init__()
        self.mu = float(mu)
        self.P  = nn.Embedding(n_u, k); self.Q = nn.Embedding(n_m, k)
        self.bu = nn.Embedding(n_u, 1); self.bi = nn.Embedding(n_m, 1)
        nn.init.normal_(self.P.weight, std=0.01); nn.init.normal_(self.Q.weight, std=0.01)
        nn.init.zeros_(self.bu.weight); nn.init.zeros_(self.bi.weight)
    def forward(self, u, m):
        dot = (self.P(u) * self.Q(m)).sum(1)
        return self.mu + self.bu(u).squeeze(1) + self.bi(m).squeeze(1) + dot

model = MF(n_users, n_movies, N_FACTORS, mu).to(device)
# Regularization goes in the LOSS (Funk-SVD style), NOT in Adam's weight_decay.
# weight_decay=lambda over a *mean* MSE loss applies lambda*p at full strength every
# step while the per-batch data gradient is ~error/batch_size (tiny) -> decay wins,
# params freeze at 0, and RMSE sticks at the rating std (~1.08). Explicit L2 on the
# rows used in each batch is mean-scaled to match the data term, so it regularizes
# without freezing.
opt = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.MSELoss()

u_t = torch.as_tensor(u_tr, dtype=torch.long)
m_t = torch.as_tensor(m_tr, dtype=torch.long)
r_t = torch.as_tensor(r_tr, dtype=torch.float32)
n = len(r_t)

for epoch in range(N_EPOCHS):
    perm = torch.randperm(n)
    t0 = time.time(); tot = 0.0
    model.train()
    for i in range(0, n, BATCH):
        idx = perm[i:i + BATCH]
        bu_, bm_, br_ = u_t[idx].to(device), m_t[idx].to(device), r_t[idx].to(device)
        opt.zero_grad()
        pred = model(bu_, bm_)
        mse = loss_fn(pred, br_)
        reg = REG * (model.P(bu_).pow(2).sum(1).mean() + model.Q(bm_).pow(2).sum(1).mean()
                     + model.bu(bu_).pow(2).mean() + model.bi(bm_).pow(2).mean())
        loss = mse + reg
        loss.backward(); opt.step()
        tot += mse.item() * len(idx)              # track data RMSE only (exclude reg)
    print(f'epoch {epoch+1:2d}  train RMSE {np.sqrt(tot/n):.4f}  ({time.time()-t0:.0f}s)')

In [ ]:
@torch.no_grad()
def predict_mf(u, m, bs=1_000_000):
    model.eval(); out = np.empty(len(u), np.float32)
    for i in range(0, len(u), bs):
        bu_ = torch.as_tensor(u[i:i+bs], dtype=torch.long, device=device)
        bm_ = torch.as_tensor(m[i:i+bs], dtype=torch.long, device=device)
        out[i:i+bs] = model(bu_, bm_).clamp(1, 5).cpu().numpy()
    return out

pred_mf_te = predict_mf(u_te, m_te)
print('MF test RMSE:', rmse(pred_mf_te, r_te))

## 6. Ranking evaluation — MAP@10

Two variants, reported side by side:

- **Held-out only** — rank each user's held-out items among themselves. Cheap, but *inflated*: with mean rating ≈ 3.6 most held-out items clear the 3.5 bar, so the task is easy.
- **Sampled negatives (report this one)** — rank each user's held-out items against `N_NEG` random unseen movies (NCF-style protocol). An item is **relevant if its actual rating ≥ 3.5**; sampled negatives are non-relevant. This is the honest catalog-style Top-K metric.

`AP@10 = (1/min(10, R)) · Σ precision@i · rel_i`, averaged over users with ≥1 relevant held-out item.

> Note: a bias-only baseline ranks every candidate for a given user purely by *item bias* (`μ + b_u` is constant across a user's candidates) — i.e. pure popularity. So its sampled-negative MAP@10 is the popularity floor that the MF personalization term `pᵤ·qᵢ` must beat.

In [ ]:
def map_at_k(u_te, m_te, r_te, predict_fn, k=10, max_users=20000, seed=SEED):
    """Held-out-only MAP@10: rank each user's held-out items among themselves."""
    pred = predict_fn(u_te, m_te)
    o = np.argsort(u_te, kind='stable')
    uu, rr, pp = u_te[o], r_te[o], pred[o]
    users_u, starts = np.unique(uu, return_index=True)
    ends = np.append(starts[1:], len(uu))
    rgn = np.random.default_rng(seed)
    sel = np.arange(len(users_u))
    if len(sel) > max_users:
        sel = rgn.choice(sel, max_users, replace=False)
    aps = []
    for j in sel:
        s, e = starts[j], ends[j]
        if e - s < 2:
            continue
        rel = rr[s:e] >= RELEVANT_THRESHOLD
        R = int(rel.sum())
        if R == 0:
            continue
        rank = np.argsort(-pp[s:e], kind='stable')[:k]
        hits = 0; ap = 0.0
        for i, ri in enumerate(rank, 1):
            if rel[ri]:
                hits += 1; ap += hits / i
        aps.append(ap / min(k, R))
    return float(np.mean(aps)), len(aps)


def map_at_k_sampled(predict_fn, u_tr, m_tr, u_te, m_te, r_te,
                     k=10, n_neg=100, max_users=10000, seed=SEED):
    """Honest Top-K eval: rank each user's held-out items against n_neg random
    unseen movies. Relevant = held-out item with rating >= RELEVANT_THRESHOLD."""
    rgn = np.random.default_rng(seed)
    o = np.argsort(u_tr, kind='stable')                 # train movies by user (CSR-style)
    su, sm = u_tr[o], m_tr[o]
    bounds = np.searchsorted(su, np.arange(n_users + 1))
    ot = np.argsort(u_te, kind='stable')                # test grouped by user
    tu, tm, trr = u_te[ot], m_te[ot], r_te[ot]
    tusers, tst = np.unique(tu, return_index=True)
    ten = np.append(tst[1:], len(tu))
    sel = np.arange(len(tusers))
    if len(sel) > max_users:
        sel = rgn.choice(sel, max_users, replace=False)
    aps = []
    for j in sel:
        uidx = int(tusers[j]); s, e = tst[j], ten[j]
        t_movies, t_r = tm[s:e], trr[s:e]
        rel = t_r >= RELEVANT_THRESHOLD
        R = int(rel.sum())
        if R == 0:
            continue
        seen = set(sm[bounds[uidx]:bounds[uidx + 1]].tolist()) | set(t_movies.tolist())
        negs = []
        while len(negs) < n_neg:                        # rejection-sample unseen movies
            for c in rgn.integers(0, n_movies, n_neg):
                c = int(c)
                if c not in seen:
                    negs.append(c); seen.add(c)
                    if len(negs) >= n_neg:
                        break
        cand = np.concatenate([t_movies, np.array(negs, dtype=np.int32)])
        labels = np.concatenate([rel, np.zeros(len(negs), bool)])
        scores = predict_fn(np.full(len(cand), uidx, np.int32), cand)
        rank = np.argsort(-scores, kind='stable')[:k]
        hits = 0; ap = 0.0
        for i, ri in enumerate(rank, 1):
            if labels[ri]:
                hits += 1; ap += hits / i
        aps.append(ap / min(k, R))
    return float(np.mean(aps)), len(aps)


for name, fn in [('baseline', predict_baseline), ('MF', predict_mf)]:
    s_ho, n_ho = map_at_k(u_te, m_te, r_te, fn)
    s_sn, n_sn = map_at_k_sampled(fn, u_tr, m_tr, u_te, m_te, r_te)
    print(f'{name:9s} MAP@10  held-out={s_ho:.4f} ({n_ho})  '
          f'sampled-neg={s_sn:.4f} ({n_sn})')

## 7. Top-K recommendations + qualitative examples

`top_k_for_user` supports a **temperature** knob for diversity:
- `temperature=0` → greedy (the K highest-scoring unseen movies; deterministic, max precision).
- `temperature>0` → sample K without replacement from `softmax(score/temperature)`. Higher
  temperature spreads probability toward lower-scored items, trading a little precision for
  serendipity/coverage — useful when the greedy list is repetitive (e.g. a franchise blob).

The cell prints greedy vs `0.3` vs `0.8` for one heavy rater so you can see the diversity grow.

In [ ]:
@torch.no_grad()
def top_k_for_user(uidx, k=10, exclude_seen=True, temperature=0.0):
    """Top-K by MF score.
    temperature=0 -> greedy (deterministic argmax top-k).
    temperature>0 -> sample k items WITHOUT replacement from softmax(score/temperature),
    injecting diversity/serendipity. Higher temp = more exploratory (less precise)."""
    model.eval()
    allm = torch.arange(n_movies, device=device)
    uvec = torch.full((n_movies,), uidx, dtype=torch.long, device=device)
    scores = model(uvec, allm).cpu().numpy()
    if exclude_seen:
        scores[m_tr[u_tr == uidx]] = -np.inf
    if temperature <= 0:
        top = np.argpartition(-scores, k)[:k]
    else:
        logits = scores / temperature
        logits -= np.nanmax(logits[np.isfinite(logits)])      # stabilize; -inf stays -inf
        p = np.exp(logits); p[~np.isfinite(p)] = 0.0; p /= p.sum()
        top = rng.choice(n_movies, size=k, replace=False, p=p)
    top = top[np.argsort(-scores[top])]                        # show best-first
    return [(title_of.get(mid + 1, f'movie {mid+1}'), float(scores[mid])) for mid in top]

uidx = int(np.argmax(user_counts))        # a heavy rater
print('Liked (train, rating 5):')
for mid in m_tr[(u_tr == uidx) & (r_tr == 5)][:8]:
    print('  -', title_of.get(mid + 1, mid + 1))

for temp in (0.0, 0.3, 0.8):              # greedy vs increasingly diverse
    print(f"\nTop-10 ({'greedy' if temp == 0 else f'temperature={temp}'}):")
    for t, s in top_k_for_user(uidx, temperature=temp):
        print(f'  {s:.2f}  {t}')

## 8. Ensemble blend (ridge over model predictions)

The Prize lesson: blends beat any single model. Fit a small linear model on a held-out set over `[baseline, MF, ...]` predictions. Add kNN / SVD++ columns as you build them. For a clean blend, fit on `probe.txt` rather than the same test set.

In [ ]:
from sklearn.linear_model import Ridge
X = np.column_stack([predict_baseline(u_te, m_te), pred_mf_te])   # + kNN, SVD++ cols
half = len(r_te) // 2
blend = Ridge(alpha=1.0).fit(X[:half], r_te[:half])               # fit on one half...
pred_blend = np.clip(blend.predict(X[half:]), 1, 5)               # ...eval on the other
print('blend weights:', blend.coef_, 'intercept', blend.intercept_)
print('blend test RMSE:', rmse(pred_blend, r_te[half:]))

## 9. Extensions (stubs / notes)

- **Item-item kNN** — second model for the mandatory comparison. Cosine/Pearson on mean-centered ratings with shrinkage `s_ij * n_ij/(n_ij+λ)`; predict from top-N neighbors the user rated. Captures local patterns MF smooths over. Add its prediction as a blend column.
- **`implicit` ALS** — `pip install implicit`; fast Cython ALS. Alternative latent-factor optimizer for the comparison.
- **SVD++ / timeSVD++** — add implicit feedback `|N(u)|^{-1/2} Σ y_j` and time-dependent biases `b_u(t)`, `b_i(t)`. The EDA temporal plot motivates this; historically reached RMSE ≈ 0.876.
- **BPR / WARP** — to optimize ranking (sampled-neg MAP@10) directly rather than via rating accuracy. The baseline's popularity-floor MAP@10 shows how much headroom ranking-aware training has.